In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dbx_projects;
USE CATALOG dbx_projects;

CREATE SCHEMA IF NOT EXISTS landing;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;
CREATE SCHEMA IF NOT EXISTS control;
CREATE SCHEMA IF NOT EXISTS quarantine;

CREATE VOLUME IF NOT EXISTS landing.files;

In [0]:
from pyspark.sql import functions as F
import json, random
from datetime import date, timedelta

In [0]:
CATALOG = "dbx_projects"
VOLUME  = f"/Volumes/{CATALOG}/landing/files"

In [0]:
def generate_day(run_date:str, n_orders:int=2000,inject_bad:bool=True):
    random.seed(hash(run_date)%10000)
    rows=[]
    for i in range(n_orders):
        rows.append({
            "ord_id":str(abs(hash(run_date + str(i)))%10**8),
            "cust":f" {random.randint(1,500)} ",
            "amt":f"{round(random.uniform(5,800),2)}",
            "dt":"/".join(reversed(run_date.split("-"))),
            "status":random.choice(["completed", "COMPLETED", "cancelled", "pending"])
        })

    if inject_bad:
        rows += [
            {"ord_id": None, "cust": "7", "amt": "50.00", "dt": "/".join(reversed(run_date.split("-"))), "status": "completed"},
            {"ord_id": "99999901", "cust": "8", "amt": "-25.00", "dt": "/".join(reversed(run_date.split("-"))), "status": "completed"},
            {"ord_id": "99999902", "cust": "9", "amt": "N/A", "dt": "/".join(reversed(run_date.split("-"))), "status": "completed"},
            ]
        
    dbutils.fs.mkdirs(f"{VOLUME}/orders/")
    dbutils.fs.put(f"{VOLUME}/orders/orders_{run_date}.json", "\n".join(json.dumps(r) for r in rows), overwrite=True)


In [0]:
customers = "\n".join(
    ["customer_id, name, country, segment,"] +
    [f"{i}, Customer_{i}, {'IN' if i%3 else 'US' if i%2 else 'UK'},"
     f"{'PREMIUM' if i%5==0 else 'STANDARD'}" for i in range(1, 501)]
)

dbutils.fs.mkdirs(f"{VOLUME}/customers/")
dbutils.fs.put(f"{VOLUME}/customers/customers.csv", customers, overwrite=True)

In [0]:
for d in range(5,0,-1):
    generate_day(str(date.today() - timedelta(days=d)))

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbx_projects")
dbutils.widgets.text("entity", "orders")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
ENTITY = dbutils.widgets.get("entity")

FMT = "csv" if ENTITY =="customers" else "json"

In [0]:
reader = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", FMT)
          .option("cloudFiles.schemaLocation", f"/Volumes/{CATALOG}/landing/files/_schema/{ENTITY}")
          .option("cloudFiles.schemaEvolutionMode", "rescue")
          .option("cloudFiles.maxFilesPerTrigger", 100)
          .option("cloudFiles.inferColumnTypes", "false")
          .option("pathGlobFIlter", f"*.{FMT}"))

In [0]:
if FMT == 'csv':
    reader = reader.option("header", "true").option("mode", "PERMISSIVE")

In [0]:
df = (reader.load(f"/Volumes/{CATALOG}/landing/files/{ENTITY}")
      .withColumn("_ingested_at", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path"))
      .withColumn("_file_modified", F.col("_metadata.file_modification_time"))
      .withColumn("_ingested_date", F.current_date()))

In [0]:
(df.writeStream
 .option("checkpointLocation", f"/Volumes/{CATALOG}/landing/files/_ckpt/{ENTITY}")
 .option("mergeSchema", "true")
 .trigger(availableNow=True)
 .toTable(f"{CATALOG}.bronze.{ENTITY}_raw")
 .awaitTermination())